In [1]:
# Install the Kaggle API client
!pip install kaggle

# Upload your kaggle.json file to Colab
from google.colab import files
files.upload()

# Create the .kaggle directory and copy the API token
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset from Kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

# Unzip the downloaded files
!unzip -q skin-cancer-mnist-ham10000.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [01:30<00:00, 69.6MB/s]
100% 5.20G/5.20G [01:30<00:00, 61.5MB/s]


In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# The dataset is now in the working directory
data_dir = './'
image_dir = './images'
metadata_path = os.path.join(data_dir, 'HAM10000_metadata.csv')

# Load and preprocess the metadata
df_meta = pd.read_csv(metadata_path)
df_meta['path'] = df_meta['image_id'].apply(lambda x: os.path.join(image_dir, x + '.jpg'))
df_meta['label'] = df_meta['dx']
num_classes = len(df_meta['label'].unique())
print(f"Number of classes: {num_classes}")

# Split data into training and testing sets
train_df, test_df = train_test_split(df_meta, test_size=0.2, random_state=42, stratify=df_meta['label'])

# Image dimensions and batch size
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

# Create data generators with augmentation for the training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Number of classes: 7
Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames belonging to 0 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 8012 invalid image filename(s) in x_col="path". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 2003 invalid image filename(s) in x_col="path". These filename(s) will be ignored.
  warnings.warn(


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build the Sequential model
model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D((2, 2)),

    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Flatten the output for the Dense layers
    Flatten(),

    # Dropout layer to prevent overfitting
    Dropout(0.5),

    # Fully connected layers
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax') # Output layer
])

# Compile the model
model.compile(optimizer='Adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,543 (12.61 MB)

 Trainable params: 3,305,543 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# Cell 1: Setup and Data Download (No changes needed here)
# ... (run your previous code for downloading the data)

#--------------------------------------------------------------------------

# Cell 2: Data Preprocessing and Generators (Corrected)
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# The dataset is now in the working directory
data_dir = './'
# Corrected image directory path to point to the 'images' folder
image_dir = './HAM10000_images_part_1'
# The HAM10000 dataset is split into several folders. We need to check
# the unzipped folder names. The first one is typically 'HAM10000_images_part_1'.

metadata_path = os.path.join(data_dir, 'HAM10000_metadata.csv')

# Load and preprocess the metadata
df_meta = pd.read_csv(metadata_path)
# Corrected path creation to account for the unzipped folder structure
df_meta['path'] = df_meta['image_id'].apply(lambda x: os.path.join(image_dir, x + '.jpg'))
df_meta['label'] = df_meta['dx']
num_classes = len(df_meta['label'].unique())
print(f"Number of classes: {num_classes}")

# Split data into training and testing sets
train_df, test_df = train_test_split(df_meta, test_size=0.2, random_state=42, stratify=df_meta['label'])

# Image dimensions and batch size
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

# Create data generators with augmentation for the training set
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

try:
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='path',
        y_col='label',
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='path',
        y_col='label',
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
except Exception as e:
    print(f"Error during data generator creation: {e}")
    print("Please check the 'image_dir' path. A list of files in the current directory might help:")
    !ls -R

#--------------------------------------------------------------------------

# Cell 3: Build and Compile the CNN Model (No changes needed)
# ... (your existing code for building the model)

#--------------------------------------------------------------------------

# Cell 4: Train the Model (Now this should work correctly)
from sklearn.utils import class_weight

# Calculate class weights to handle class imbalance
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights_dict = dict(enumerate(class_weights))

# Train the model
EPOCHS = 20
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    class_weight=class_weights_dict
)

Number of classes: 7
Found 4010 validated image filenames belonging to 7 classes.
Found 990 validated image filenames belonging to 7 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 4002 invalid image filename(s) in x_col="path". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 1013 invalid image filename(s) in x_col="path". These filename(s) will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 73s 494ms/step - accuracy: 0.3586 - loss: 1.8784 - val_accuracy: 0.3394 - val_loss: 1.8612
Epoch 2/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 361ms/step - accuracy: 0.3517 - loss: 1.6966 - val_accuracy: 0.4394 - val_loss: 1.6955
Epoch 3/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 354ms/step - accuracy: 0.4381 - loss: 1.7772 - val_accuracy: 0.3576 - val_loss: 1.7555
Epoch 4/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 359ms/step - accuracy: 0.3123 - loss: 1.8722 - val_accuracy: 0.4939 - val_loss: 1.3461
Epoch 5/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 354ms/step - accuracy: 0.4716 - loss: 1.6151 - val_accuracy: 0.5212 - val_loss: 1.4173
Epoch 6/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 359ms/step - accuracy: 0.4340 - loss: 1.6623 - val_accuracy: 0.3404 - val_loss: 1.7892
Epoch 7/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 81s 355ms/step - accuracy: 0.3526 - loss: 1.6437 - val_accuracy: 0.4414 - val_loss: 1.4945
Epoch 8/20
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 358ms/step - accuracy: 0.4173 - loss: 1

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# The dataset is now in the working directory
data_dir = './'
metadata_path = os.path.join(data_dir, 'HAM10000_metadata.csv')

# Load and preprocess the metadata
df_meta = pd.read_csv(metadata_path)

# ----------------- FIX FOR FILE NOT FOUND ERROR -----------------

# List all image directories
image_dirs = [d for d in os.listdir(data_dir) if d.startswith('HAM10000_images')]

# Create a dictionary to map image IDs to their full paths
image_path_dict = {}
for img_dir in image_dirs:
    full_path = os.path.join(data_dir, img_dir)
    for img_file in os.listdir(full_path):
        image_id = os.path.splitext(img_file)[0]
        image_path_dict[image_id] = os.path.join(full_path, img_file)

# Update the 'path' column in the dataframe using the dictionary
df_meta['path'] = df_meta['image_id'].map(image_path_dict)
df_meta['label'] = df_meta['dx']
num_classes = len(df_meta['label'].unique())
print(f"Number of classes: {num_classes}")

# ----------------- END OF FIX -----------------

# Split data into training and testing sets
train_df, test_df = train_test_split(df_meta, test_size=0.2, random_state=42, stratify=df_meta['label'])

# Image dimensions and batch size
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32

# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Number of classes: 7
Found 8012 validated image filenames belonging to 7 classes.
Found 2003 validated image filenames belonging to 7 classes.


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np

# A function to make a single prediction
def predict_image(model, img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Create a batch
    img_array /= 255.0 # Normalize pixel values

    prediction = model.predict(img_array)
    class_labels = list(train_generator.class_indices.keys())
    predicted_class_index = np.argmax(prediction)
    predicted_class = class_labels[predicted_class_index]
    confidence = np.max(prediction)

    return predicted_class, confidence, img

# Define UI components
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)
predict_button = widgets.Button(
    description='Predict',
    disabled=True,
    button_style='success',
    tooltip='Click to predict the uploaded image',
    icon='check'
)
output = widgets.Output()

# Handle file upload event
def on_upload_change(change):
    if uploader.value:
        predict_button.disabled = False
        with output:
            clear_output(wait=True)
            print("Image uploaded. Click 'Predict' to get a diagnosis.")
    else:
        predict_button.disabled = True
        with output:
            clear_output(wait=True)

# Handle button click event
def on_predict_click(b):
    if not uploader.value:
        with output:
            clear_output()
            print("Please upload an image first.")
        return

    # Get the uploaded image data
    for name, file_info in uploader.value.items():
        image_data = file_info['content']
        img = Image.open(io.BytesIO(image_data))

        # This is the key part to handle PNG files.
        # Convert the image to RGB if it's in RGBA mode
        if img.mode == 'RGBA':
            img = img.convert('RGB')

        # Now save the image as a JPEG
        img.save('uploaded_image.jpg')
        img_path = 'uploaded_image.jpg'

        # Make prediction
        predicted_class, confidence, original_img = predict_image(model, img_path)

        # Display results in the output widget
        with output:
            clear_output(wait=True)
            plt.imshow(original_img)
            plt.title(f"Predicted: {predicted_class} (Confidence: {confidence:.2f})")
            plt.show()
            print(f"Prediction: {predicted_class}")
            print(f"Confidence: {confidence:.2f}")

            # Warning for dangerous conditions
            dangerous_conditions = ['melanoma', 'basal cell carcinoma', 'actinic keratoses']
            if predicted_class.lower() in dangerous_conditions:
                print("\n🚨 **WARNING**: This condition is potentially dangerous. Please consult a dermatologist.")
            else:
                print("\n✅ This condition is likely benign, but a medical opinion is always recommended.")

# Link events to handlers
uploader.observe(on_upload_change, names='value')
predict_button.on_click(on_predict_click)

# Display the widgets
display(uploader, predict_button, output)

FileUpload(value={}, accept='image/*', description='Upload Image')

Button(button_style='success', description='Predict', disabled=True, icon='check', style=ButtonStyle(), toolti…

Output()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
